# 05 — Feature Engineering & Model Selection Plan (Phase 3)

This notebook delivers the Phase 3 requirements:

1. **≥10 engineered features**, each documented with source columns,
   formula, rationale, missing-value handling, and eligibility per
   prediction time.
2. A **task-to-model plan** for regression and classification, with
   **justification** for each candidate model.
3. A **validation strategy** justified against the chronological split
   defined in Notebook 01.
4. A **computational budget** and **reproducible sampling** plan.
5. A bridge to Phase 4 (model-building notebooks).

All engineered features are deterministic. Learned transformations
(imputation, scaling, encoding, clipping) remain inside sklearn Pipelines
and will be fit on the training folds only in Phase 4.

In [1]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import LendingClubFeatureEngineer
from src.splits import ChronologicalSplitter

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIG = PROJECT_ROOT / 'outputs' / 'figures'; OUT_FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Reuse the frozen chronological split

The split definition was frozen in Notebook 01 and used consistently in
Notebooks 02–04. Phase 3 reuses it verbatim: same cutoffs, same rows.

In [2]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

In [3]:
splitter = ChronologicalSplitter(**split_cfg)

acc_clf = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification.parquet')
acc_reg = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression.parquet')
print('acc_clf:', acc_clf.shape)
print('acc_reg:', acc_reg.shape)

splits_clf = splitter.split(acc_clf)
splits_reg = splitter.split(acc_reg)

print('CLF sizes (train/val/test):',
      len(splits_clf.train), len(splits_clf.val), len(splits_clf.test))
print('REG sizes (train/val/test):',
      len(splits_reg.train), len(splits_reg.val), len(splits_reg.test))

acc_clf: (1379602, 152)
acc_reg: (2260668, 152)
CLF sizes (train/val/test): 451671 678142 249789
REG sizes (train/val/test): 466345 855502 938821


## 2. Define eligibility lists for each task

**Regression cohort** (pre-pricing, target = `int_rate`): all pre-application
credit-profile variables, plus engineered features that do not use
`installment`, `int_rate`, `grade`, `sub_grade`.

**Classification cohort** (origination, target = 3-class outcome): the same
plus origination-time priced fields (`int_rate`, `installment`, `grade`,
`sub_grade` are allowed by the specification because they are fixed at
origination and not derived from the repayment outcome).

In [4]:
NUM_REG = [
    'loan_amnt', 'term', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec',
    'pub_rec_bankruptcies', 'mort_acc', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy',
    'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'pct_tl_nvr_dlq',
]
NUM_CLF = NUM_REG + ['installment', 'int_rate']  # allowed at origination

CAT_COLS = ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'application_type']

NUM_REG = [c for c in NUM_REG if c in acc_reg.columns]
NUM_CLF = [c for c in NUM_CLF if c in acc_clf.columns]
CAT_COLS = [c for c in CAT_COLS if c in acc_clf.columns]

print('NUM_REG:', len(NUM_REG))
print('NUM_CLF:', len(NUM_CLF))
print('CAT_COLS:', len(CAT_COLS))

NUM_REG: 22
NUM_CLF: 24
CAT_COLS: 5


## 3. Build engineered features — regression task

We apply the feature engineer to the regression dataframe, then
re-split so the engineered columns are present in train/val/test.

In [5]:
engineer_reg = LendingClubFeatureEngineer()   # per-row issue_d as reference
acc_reg_fe = engineer_reg.transform(acc_reg, task='regression')

spec_reg = engineer_reg.specification_frame()
spec_reg.to_csv(OUT_TABLES / 'feature_engineering_spec_regression.csv', index=False)
spec_reg[['name','source_columns','formula','missing_handling',
          'eligible_regression','eligible_classification']]

,name,source_columns,formula,missing_handling,eligible_regression,eligible_classification
0,loan_to_income,"loan_amnt, annual_inc",loan_amnt / annual_inc,NaN preserved; imputed inside the training-fol...,True,True
1,dti_x_loan_to_income,"dti, loan_to_income",dti * loan_to_income,NaN preserved; imputed inside the training-fol...,True,True
2,credit_hist_months,"earliest_cr_line, issue_d",(reference_date - earliest_cr_line) in months,NaN preserved; imputed inside the training-fol...,True,True
3,revol_util_bucket,revol_util,"discretised revol_util into [<30, 30-60, 60-80...",NaN preserved; imputed inside the training-fol...,True,True
4,total_credit_lines,"open_acc, total_acc",open_acc + total_acc,NaN preserved; imputed inside the training-fol...,True,True
5,inq_per_year,inq_last_6mths,inq_last_6mths * 2,NaN preserved; imputed inside the training-fol...,True,True
6,util_x_fico,"revol_util, fico_range_low",revol_util * fico_range_low,NaN preserved; imputed inside the training-fol...,True,True
7,bc_pressure,"bc_util, bc_open_to_buy","bc_util / max(bc_open_to_buy, eps)",NaN preserved; imputed inside the training-fol...,True,True
8,delinq_rate,"delinq_2yrs, total_acc","delinq_2yrs / max(total_acc, 1)",NaN preserved; imputed inside the training-fol...,True,True
9,loan_per_term,"loan_amnt, term",loan_amnt / term,NaN preserved; imputed inside the training-fol...,True,True


## 4. Build engineered features — classification task

Same features, plus the two that are allowed at origination but excluded
from the pre-pricing regression (`installment_burden`, `rate_burden`).

In [6]:
engineer_clf = LendingClubFeatureEngineer()
acc_clf_fe = engineer_clf.transform(acc_clf, task='classification')

spec_clf = engineer_clf.specification_frame()
spec_clf.to_csv(OUT_TABLES / 'feature_engineering_spec_classification.csv', index=False)
spec_clf[['name','source_columns','formula','missing_handling',
          'eligible_regression','eligible_classification']]

,name,source_columns,formula,missing_handling,eligible_regression,eligible_classification
0,loan_to_income,"loan_amnt, annual_inc",loan_amnt / annual_inc,NaN preserved; imputed inside the training-fol...,True,True
1,dti_x_loan_to_income,"dti, loan_to_income",dti * loan_to_income,NaN preserved; imputed inside the training-fol...,True,True
2,credit_hist_months,"earliest_cr_line, issue_d",(reference_date - earliest_cr_line) in months,NaN preserved; imputed inside the training-fol...,True,True
3,revol_util_bucket,revol_util,"discretised revol_util into [<30, 30-60, 60-80...",NaN preserved; imputed inside the training-fol...,True,True
4,total_credit_lines,"open_acc, total_acc",open_acc + total_acc,NaN preserved; imputed inside the training-fol...,True,True
5,inq_per_year,inq_last_6mths,inq_last_6mths * 2,NaN preserved; imputed inside the training-fol...,True,True
6,util_x_fico,"revol_util, fico_range_low",revol_util * fico_range_low,NaN preserved; imputed inside the training-fol...,True,True
7,bc_pressure,"bc_util, bc_open_to_buy","bc_util / max(bc_open_to_buy, eps)",NaN preserved; imputed inside the training-fol...,True,True
8,delinq_rate,"delinq_2yrs, total_acc","delinq_2yrs / max(total_acc, 1)",NaN preserved; imputed inside the training-fol...,True,True
9,loan_per_term,"loan_amnt, term",loan_amnt / term,NaN preserved; imputed inside the training-fol...,True,True


## 5. Combined specification table (the deliverable)

The union of the two specs is our **≥10 engineered-feature specification**.
Each row records source columns, formula, rationale, missing-value handling,
and eligibility per task — exactly as the specification requires.

In [7]:
spec_all = pd.concat([spec_reg, spec_clf]).drop_duplicates(subset=['name']).reset_index(drop=True)
spec_all.to_csv(OUT_TABLES / 'feature_engineering_spec.csv', index=False)

print('Total engineered features:', len(spec_all))
spec_all[['name','formula','eligible_regression','eligible_classification']]

Total engineered features: 15


,name,formula,eligible_regression,eligible_classification
0,loan_to_income,loan_amnt / annual_inc,True,True
1,dti_x_loan_to_income,dti * loan_to_income,True,True
2,credit_hist_months,(reference_date - earliest_cr_line) in months,True,True
3,revol_util_bucket,"discretised revol_util into [<30, 30-60, 60-80...",True,True
4,total_credit_lines,open_acc + total_acc,True,True
5,inq_per_year,inq_last_6mths * 2,True,True
6,util_x_fico,revol_util * fico_range_low,True,True
7,bc_pressure,"bc_util / max(bc_open_to_buy, eps)",True,True
8,delinq_rate,"delinq_2yrs / max(total_acc, 1)",True,True
9,loan_per_term,loan_amnt / term,True,True


## 6. Quick visual check of engineered features

We plot a few engineered features on the classification cohort to confirm
they are well-scaled and informative. These are **descriptive only** — no
model training happens in this notebook.

In [8]:
feature_to_plot = 'loan_to_income'
if feature_to_plot in acc_clf_fe.columns:
    data = acc_clf_fe[feature_to_plot].dropna()
    data = data[data.between(data.quantile(0.01), data.quantile(0.99))]  # trim outliers for the plot
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(data, bins=60, color='steelblue', edgecolor='white')
    ax.set_title(f'{feature_to_plot} (1st-99th percentile)')
    ax.set_xlabel(feature_to_plot); ax.set_ylabel('count')
    fig.tight_layout()
    fig.savefig(OUT_FIG / f'fe_hist_{feature_to_plot}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('saved histogram for', feature_to_plot)

saved histogram for loan_to_income


In [9]:
feature_to_plot = 'util_x_fico'
if feature_to_plot in acc_clf_fe.columns:
    data = acc_clf_fe[feature_to_plot].dropna()
    data = data[data.between(data.quantile(0.01), data.quantile(0.99))]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(data, bins=60, color='darkorange', edgecolor='white')
    ax.set_title(f'{feature_to_plot} (1st-99th percentile)')
    ax.set_xlabel(feature_to_plot); ax.set_ylabel('count')
    fig.tight_layout()
    fig.savefig(OUT_FIG / f'fe_hist_{feature_to_plot}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('saved histogram for', feature_to_plot)

saved histogram for util_x_fico


## 7. Missing-value audit of engineered features

Missing values in engineered features come from three sources:
1. Missing inputs (`annual_inc`, `earliest_cr_line`, etc.).
2. Division-by-zero guards (mapped to NaN, not 0).
3. Non-applicability (not relevant here — no `*_joint` features were used).

All missingness will be handled **inside** the training-fold pipeline in
Phase 4 via `SimpleImputer`.

In [10]:
engineered_names = spec_all['name'].tolist()
engineered_present = [c for c in engineered_names if c in acc_clf_fe.columns]

missing_table = pd.DataFrame({
    'missing': acc_clf_fe[engineered_present].isna().sum(),
    'missing_pct': (acc_clf_fe[engineered_present].isna().mean() * 100).round(3),
}).sort_values('missing_pct', ascending=False)
missing_table.to_csv(OUT_TABLES / 'feature_engineering_missingness.csv')
missing_table

,missing,missing_pct
bc_pressure,84432,6.120
emp_length_years,81403,5.900
revol_util_bucket,895,0.065
util_x_fico,895,0.065
dti_x_loan_to_income,412,0.030
loan_to_income,396,0.029
installment_burden,396,0.029
credit_hist_months,0,0.000
total_credit_lines,0,0.000
inq_per_year,1,0.000


## 8. Save feature-enriched datasets

Phase 4 will load these directly, so the engineered columns and the
chronological split are guaranteed to match what we defined here.

In [11]:
acc_reg_fe.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression_fe.parquet', index=False)
acc_clf_fe.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification_fe.parquet', index=False)
print('saved enriched regression and classification parquet files')

saved enriched regression and classification parquet files


## 9. Task-to-model plan (Phase 3 deliverable)

For each task, we state the candidate models and justify each choice. Only
*candidate* models are listed here; actual training, tuning, and comparison
happen in Phase 4.

### Regression — target `int_rate`

| Model | Why | Notes |
|---|---|---|
| **kNN (from scratch, NumPy)** | Required by the specification. Establishes a non-parametric baseline. | Sensitive to feature scale; all numerics must be standardised inside the pipeline. |
| **Ridge Regression (scikit-learn)** | Linear baseline with L2 regularisation, robust to multicollinearity. | Directly interpretable coefficients; useful for feature-importance story. |
| **Decision Tree Regressor (scikit-learn)** | Non-linear baseline; also a building block for the ensemble section. | Prone to overfitting; depth and min-samples-leaf tuned via CV. |
| **Random Forest Regressor (bagging ensemble)** | Handles non-linearity, mixed feature types, and interactions without hand-engineering. | Required as the bagging ensemble. |
| **Gradient Boosting Regressor (boosting ensemble)** | State-of-the-art on tabular data; strong on interactions and outliers. | Required as the boosting ensemble. |
| **MLP (PyTorch or TensorFlow)** | Captures complex non-linear structure and can share representation with the classifier. | Justified below. |

### Classification — target `target_class ∈ {0,1,2}`

| Model | Why | Notes |
|---|---|---|
| **kNN (from scratch)** | Required. Non-parametric 3-class baseline. | Same standardisation requirement. |
| **Logistic Regression (multinomial)** | Linear baseline; calibrated probabilities. | Class imbalance handled via `class_weight='balanced'`. |
| **Decision Tree Classifier** | Non-linear baseline; interpretable rules. | Class-balanced splits via `class_weight`. |
| **Random Forest Classifier (bagging)** | Strong on tabular data; handles imbalance with class weights. | Required bagging ensemble. |
| **Gradient Boosting Classifier (boosting)** | Typically the best single model on tabular classification. | Required boosting ensemble. |
| **MLP (PyTorch or TensorFlow)** | Required deep-learning component. | Class-weighted cross-entropy for imbalance. |

### Justification of the MLP architecture (for both tasks)

- **Input layer:** ~30–40 features after one-hot encoding of low-cardinality categoricals.
- **Hidden layers:** two hidden layers of 128 → 64 units, ReLU activation, dropout 0.2–0.3.
- **Output:**
  - Regression: one linear unit (predicting `int_rate`).
  - Classification: three units with softmax (predicting class probabilities).
- **Why this architecture:** the tabular signal is weak and diffuse (see the effect sizes from Notebook 04 — all |δ| ≤ 0.2). A shallow MLP has enough capacity to combine many weak signals without overfitting the noise, while remaining cheap to train on the available hardware.

### Clustering (Phase 4 preview)

| Algorithm | Rationale |
|---|---|
| **k-Means** | Fast, interpretable, requires k selection via silhouette / elbow. |
| **Agglomerative (Ward linkage)** | Hierarchical structure; dendrogram aids interpretation. |
| **DBSCAN** (optional) | Density-based; finds non-convex clusters and identifies outliers. |
| **Gaussian Mixture (optional)** | Soft assignments; useful for overlapping credit profiles. |

## 10. Validation strategy

**Chronological split (frozen in Notebook 01):**

- Train: up to **2014-12**
- Validation: **2015-01 to 2016-12**
- Test: **2017-01 onward** — never touched until Phase 5.

**Within-train cross-validation:**

We use `TimeSeriesSplit` on the training portion (expanding-window folds,
5 splits) for every model that requires hyperparameter tuning. This respects
the temporal ordering and matches the eventual test scenario (predicting on
later loans).

**No resampling before splitting.** Class-imbalance techniques (class weights,
focal loss) are applied inside training folds only. The test set is never
rebalanced.

**Metrics:**

- Regression: MAE, RMSE, R². Mean and standard deviation across CV folds, plus a single value on the test set.
- Classification: confusion matrix, macro and micro precision/recall/F1, ROC-AUC (one-vs-rest), accuracy. Mean and std across CV folds, plus test-set values.
- Clustering: silhouette, Davies–Bouldin, Calinski–Harabasz, plus substantive interpretation.

## 11. Computational budget

Estimated runtimes on the development environment (Apple Silicon, 16 GB RAM):

| Model | Approx. training time (full training cohort) |
|---|---|
| kNN (from scratch, full 1.3M × 30 with tuned k) | 5–10 min per k on full data; 1–2 min on 200k subsample |
| Ridge / Logistic | seconds |
| Decision Tree | < 1 min |
| Random Forest (500 trees, depth auto) | 5–15 min |
| Gradient Boosting (500 estimators) | 10–30 min |
| MLP (2 hidden layers, 20 epochs, batch 1024) | 5–15 min per seed |

**Reproducible sampling for expensive models:**

For kNN and the MLP, we use a reproducible subsample of the training cohort
(default 200,000 rows, `np.random.default_rng(SEED)`) so that hyperparameter
tuning finishes within the wall-clock budget. The full training cohort is
used for the final fit of the cheaper models (Ridge, Logistic, trees).
Sampling is never applied to validation or test.

## 12. Reproducible sampling helper

The specification forbids using `df.head()` as a representative sample.
We use this helper consistently across phases.

In [12]:
from src.eda_utils import LendingClubEDA
eda = LendingClubEDA()

_check = eda.reproducible_sample(splits_clf.train, n=1000, seed=42)
_check.head(3)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,target_class
691526,37600434,NaN,4500.0,4500.0,4500.0,36.0,9.49,144.13,B,B2,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
691756,37720268,NaN,20000.0,20000.0,19975.0,36.0,9.49,640.57,B,B2,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
691942,37117940,NaN,14000.0,14000.0,14000.0,36.0,7.49,435.43,A,A4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0


## 13. Save the model plan

We persist the task-to-model plan as a CSV so it can be cited in the
mid-journey report and reused in Phase 4.

In [13]:
model_plan = pd.DataFrame([
    # Regression
    ('regression', 'kNN (scratch)', 'required baseline', 'non-parametric; scale-sensitive'),
    ('regression', 'Ridge', 'linear baseline', 'L2 for multicollinearity'),
    ('regression', 'DecisionTreeRegressor', 'non-linear baseline', 'depth via CV'),
    ('regression', 'RandomForestRegressor', 'bagging ensemble (required)', '500 trees'),
    ('regression', 'GradientBoostingRegressor', 'boosting ensemble (required)', '500 estimators'),
    ('regression', 'MLP (PyTorch/TensorFlow)', 'deep learning (required)', '128-64 ReLU, dropout 0.2'),
    # Classification
    ('classification', 'kNN (scratch)', 'required baseline', 'non-parametric; scale-sensitive'),
    ('classification', 'LogisticRegression', 'linear baseline', 'multinomial; class_weight=balanced'),
    ('classification', 'DecisionTreeClassifier', 'non-linear baseline', 'depth via CV; class_weight'),
    ('classification', 'RandomForestClassifier', 'bagging ensemble (required)', '500 trees; class_weight'),
    ('classification', 'GradientBoostingClassifier', 'boosting ensemble (required)', '500 estimators'),
    ('classification', 'MLP (PyTorch/TensorFlow)', 'deep learning (required)', '128-64 ReLU; weighted CE'),
    # Clustering
    ('clustering', 'KMeans', 'baseline partitional', 'k via silhouette'),
    ('clustering', 'AgglomerativeClustering', 'hierarchical', 'Ward linkage'),
    ('clustering', 'DBSCAN', 'density-based (optional)', 'eps via k-distance'),
    ('clustering', 'GaussianMixture', 'soft assignments (optional)', 'BIC for component count'),
], columns=['task','model','role','notes'])
model_plan.to_csv(OUT_TABLES / 'model_plan.csv', index=False)
model_plan

,task,model,role,notes
0,regression,kNN (scratch),required baseline,non-parametric; scale-sensitive
1,regression,Ridge,linear baseline,L2 for multicollinearity
2,regression,DecisionTreeRegressor,non-linear baseline,depth via CV
3,regression,RandomForestRegressor,bagging ensemble (required),500 trees
4,regression,GradientBoostingRegressor,boosting ensemble (required),500 estimators
5,regression,MLP (PyTorch/TensorFlow),deep learning (required),"128-64 ReLU, dropout 0.2"
6,classification,kNN (scratch),required baseline,non-parametric; scale-sensitive
7,classification,LogisticRegression,linear baseline,multinomial; class_weight=balanced
8,classification,DecisionTreeClassifier,non-linear baseline,depth via CV; class_weight
9,classification,RandomForestClassifier,bagging ensemble (required),500 trees; class_weight


## 14. What Phase 4 will do

The modelling notebooks in Phase 4 will:

1. Load `accepted_regression_fe.parquet` and `accepted_classification_fe.parquet`.
2. Rebuild the chronological split using `split_definition.json`.
3. Build a `ColumnTransformer` + `Pipeline` that fits imputation, scaling,
   and one-hot encoding on the training split only.
4. Implement kNN from scratch using NumPy arrays and a KD-tree for
   neighbour search.
5. Train and tune (via `TimeSeriesSplit` on training) all candidate models
   for regression and classification.
6. Train the ensemble models (Random Forest for bagging, Gradient Boosting
   for boosting) and the deep-learning models (MLP) for both tasks.
7. Run K-Means and at least one other clustering algorithm with varied
   hyperparameters.
8. Produce comparison tables per task (regression, classification,
   clustering) with cross-validation mean ± std and single test-set values.
9. Interpret, summarise lessons learned, and prepare the operationalisation
   plan.